# **YOLO-CIANNA SDC2 example script**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deyht/CIANNA/blob/CIANNA/examples/SKAO_SDC2/sdc2_pred_notebook.ipynb)

---


**Link to the CIANNA github repository**
https://github.com/Deyht/CIANNA

## SDC2 prediction network

This notebook is a simplified version of the scripts used in Cornu et al. 2025 ([...](...))
presenting the application of the YOLO-CIANNA 3D method over the SKAO SDC2 dataset. All network models and catalogs presented in the paper are available at https://share.obspm.fr/s/swyCT7BgEGjtZK3, along with all the scripts used to generate these results*.

This notebook **does not re-train** a network from scratch, it only load a trained model,
and do the prediction over a 3D hyperspectral cube from the SDC2. However, the MAIN cube is too large (851 GB) to be used in a Colab environment. To reproduce the main result of the paper, model trained on the LDEV cube need to be applied over the MAIN cube. For this, we recommend adapting the complete MAIN cube inference scripts provided in the code archive, and do the inference on a compatible system. Example training scripts are also provided.

Here we instead rely on the MAIN-cube trained models from Appendix E and apply them to the manageable LDEV cube. Those are the models we recommend using for transfer learning over new datasets, especially the top scoring FC-BT2 model.

The last few cells of the notebook allow to score the produced catalog using the SDC2 scorer code. These cells can be adapted to score any of the catalog provided in the archive, which allow to verify the results presented in Cornu et al. 2025 and allow easy comparison with future work.

*This is a pre-submission temporary deposit. The code and models will be archived on zenodo after the first review.


## 1- CIANNA installation


#### Query GPU allocation and properties

If nvidia-smi fail, it might indicate that you launched the colab session whithout GPU reservation.  
To change the type of reservation go to "Runtime"->"Change runtime type" and select "GPU" as your hardware accelerator.

In [1]:
%%shell

nvidia-smi

cd /content/

git clone https://github.com/NVIDIA/cuda-samples/

cd /content/cuda-samples/Samples/1_Utilities/deviceQuery/

cmake CMakeLists.txt

make SMS="50 60 70 80"

./deviceQuery | grep Capability | cut -c50- > ~/cuda_infos.txt
./deviceQuery | grep "CUDA Driver Version / Runtime Version" | cut -c57- >> ~/cuda_infos.txt

cd ~/

Mon Sep 15 14:57:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

If you are granted a GPU that does not support FP16 computation, it is advised to change the mixed precision method to FP32C_FP32A in the corresponding cells.
See the detail description on mixed precision support with CIANNA on the [Systeme Requirements](https://github.com/Deyht/CIANNA/wiki/1\)-System-Requirements) wiki page.

#### Clone CIANNA git repository

In [2]:
%%shell

cd /content/

git clone https://github.com/Deyht/CIANNA

cd CIANNA

Cloning into 'CIANNA'...
remote: Enumerating objects: 2177, done.
remote: Counting objects: 100% (606/606), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 2177 (delta 422), reused 529 (delta 370), pack-reused 1571 (from 1)
Receiving objects: 100% (2177/2177), 62.19 MiB | 35.56 MiB/s, done.
Resolving deltas: 100% (1604/1604), done.


#### Compiling CIANNA for the allocated GPU generation

There is no guaranteed forward or backward compatibility between Nvidia GPU generation, and some capabilities are generation specific. For these reasons, CIANNA must be provided the platform GPU generation at compile time.
The following cell will automatically update all the necessary files based on the detected GPU, and compile CIANNA.

In [3]:
%%shell

cd /content/CIANNA

mult="10"
cat ~/cuda_infos.txt
comp_cap="$(sed '1!d' ~/cuda_infos.txt)"
cuda_vers="$(sed '2!d' ~/cuda_infos.txt)"

lim="11.1"
old_arg=$(awk '{if ($1 < $2) print "-D CUDA_OLD";}' <<<"${cuda_vers} ${lim}")

sm_val=$(awk '{print $1*$2}' <<<"${mult} ${comp_cap}")

gen_val=$(awk '{if ($1 >= 80) print "-D GEN_AMPERE"; else if($1 >= 70) print "-D GEN_VOLTA";}' <<<"${sm_val}")

sed -i "s/.*arch=sm.*/\\t\tcuda_arg=\"\$cuda_arg -D CUDA -D comp_CUDA -lcublas -lcudart -arch=sm_$sm_val $old_arg $gen_val\"/g" compile.cp
sed -i "s/\/cuda-[0-9][0-9].[0-9]/\/cuda-$cuda_vers/g" compile.cp
sed -i "s/\/cuda-[0-9][0-9].[0-9]/\/cuda-$cuda_vers/g" src/python_module_setup.py

./compile.cp CUDA PY_INTERF

mv src/build/lib.linux-x86_64-* src/build/lib.linux-x86_64

7.5
12.5
USE_CUDA
BUILD PY_INTERF
#####  End of CUDA compilation  #####
#####  End of main program compilation  #####
#####  End of link edition and executable creation  #####
USE_CUDA
running build
running build_ext
building 'CIANNA' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -DMAX_LAYERS_NB=200 -DMAX_NETWORKS_NB=10 -DCUDA=1 -DCUDA_THREADS_PER_BLOCKS=256 -DNone -I/usr/local/cuda-12.5/include -I/usr/local/lib/python3.12/dist-packages/numpy/_core/include -I/usr/include/python3.12 -c python_module.c -o build/temp.linux-x86_64-cpython-312/python_module.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/python_module.o conv_layer.o dense_layer.o pool_layer.o norm_layer.o lrn_layer.o activ_

#### CIANNA notebook guideline

**IMPORTANT NOTE**   
CIANNA is mainly used in a script fashion and was not designed to run in notebooks. Every cell code that directly invokes CIANNA functions must be run as a script to avoid possible errors.  
To do so, the cell must have the following structure.

```
%%shell

cd /content/CIANNA

python3 - <<EOF

[... your python code ...]

EOF
```

This syntax allows one to easily edit python code in the notebook while running the cell as a script. Note that all the notebook variables can not be accessed by the cell in this context.


## 2 - Data doawnload and inference setup

The SDC2 data archive provides the LDEV cube (40 GB) and a smaller DEV cube (10GB), which is a centered cutout inside the LDEV. Downloading and normalizing these cube in a collab environement is challenging due to disk quota, RAM, and cpu limitations. Instead, we provide pre-normalized versions of these cubes. The default script makes use of the smallest dev version of the cube to keep download time reasonable, but the pre-normalized LDEV cube and the associated header can be accessed at https://share.obspm.fr/s/AYC2eZQD7sHJoZn .

The following cell create an helper python file importing the required libraries, declare variables related to the data and network setup, and declare utility functions including the data normalization function. A simplified version of the normalization function is provided for reference but not used here.
Change the **map_pixel_size_ldev** value in the following cell according to the sub version of the LDEV cube you want to use.

In [4]:
%%shell

mkdir /content/SDC2
cd /content/SDC2/

#### Download data and pre-trained models

In [9]:
%%shell

cd /content/SDC2

# Pre-normalized data download
if [ ! -f "cont_dev.fits" ]; then
  wget --content-disposition https://www.dropbox.com/scl/fo/20mzfxit0yc9djr7n56v0/AKOQA2Ex0Am4Igm6uB4Mqpo/cont_dev.fits?rlkey=mstsccclw3euwq9g71ofywjo9
fi
if [ ! -f "DEV_norm_cube.bin" ]; then
  wget --content-disposition https://share.obspm.fr/s/Y9MNGSFQ9XwnP57/download/DEV_norm_cube.bin
fi
# Trained model and metadata download
if [ ! -f "YOLO_CIANNA_net_model_SDC2_MC-BT2_MINERVA_Cornu2025.dat" ]; then
  wget --content-disposition https://share.obspm.fr/s/bjzsccaCk6NCMKp/download/YOLO_CIANNA_net_model_SDC2_MC-BT2_MINERVA_Cornu2025.dat
fi
if [ ! -f "MC-BT2_train_cat_lims.txt" ]; then
  wget --content-disposition https://share.obspm.fr/s/44wjj83qt5S8HJi/download/MC-BT2_train_cat_lims.txt
fi


--2025-09-15 15:08:49--  https://www.dropbox.com/scl/fo/20mzfxit0yc9djr7n56v0/AKOQA2Ex0Am4Igm6uB4Mqpo/cont_dev.fits?rlkey=mstsccclw3euwq9g71ofywjo9
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc5a491ea2e042155da519f7df54.dl-uk.dropboxusercontent.com/cd/0/inline/Cxa2aB2gxrt0pIaBeFK_GR39FzatoxZwLNDGqQxllDryXYN3sfh6Tmrze_9CGGmULGI5Ij27CMDXZIDWNzI7pWkgEZ5CjJc-YJckwPAcPK1Yz4bgF653xIc5JuC6XIQ5vF5351MDdYScMoManVMVsJBU/file# [following]
--2025-09-15 15:08:50--  https://uc5a491ea2e042155da519f7df54.dl-uk.dropboxusercontent.com/cd/0/inline/Cxa2aB2gxrt0pIaBeFK_GR39FzatoxZwLNDGqQxllDryXYN3sfh6Tmrze_9CGGmULGI5Ij27CMDXZIDWNzI7pWkgEZ5CjJc-YJckwPAcPK1Yz4bgF653xIc5JuC6XIQ5vF5351MDdYScMoManVMVsJBU/file
Resolving uc5a491ea2e042155da519f7df54.dl-uk.dropboxusercontent.com (uc5a491ea2e042155da519f7df54.dl-uk.dropboxuse

#### Set global configuration file

In [5]:
%%writefile /content/SDC2/config.py

#####       IMPORTS        #####

import numpy as np
import os, gc, sys, glob
from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from astropy.wcs import utils
from astropy.coordinates import SkyCoord
from numba import jit

#####   GLOBAL VARIABLES   #####

# Cube setup variables
# Data are expected to be square in the RA and DEC axes
#map_pixel_size_ldev = 1286 # For the square deg LDEV
map_pixel_size_ldev = 643 # For the 0.25 square deg DEV
map_pixel_freq_size = 6668
pixel_size = 7.77777777778E-04 #In degree
pixel_size_freq = 3.00000000000E+04 #In Hz
beam_size = 1.94444449153E-03 #In degree

# Normalization variables
do_norm = 0
cont_removal_threshold = 0.5*6e-3 #in Jy
prenorm_scaling = 0.3

# Network setup variables
sky_size = 64
freq_size = 256
nb_param = 6
nb_box = 1

# Inference setup variables
c_size_sky = 8
c_size_freq = 16
yolo_nb_sky_reg = int(sky_size/c_size_sky)
yolo_nb_freq_reg = int(freq_size/c_size_freq)

overlap_sky = c_size_sky
overlap_freq = 2*c_size_freq
patch_shift_sky = sky_size - overlap_sky
patch_shift_freq = freq_size - overlap_freq

orig_offset_sky_ldev = patch_shift_sky - ((int(map_pixel_size_ldev/2) - int(sky_size/2) + patch_shift_sky)%patch_shift_sky)
orig_offset_freq = patch_shift_freq - ((int(map_pixel_freq_size/2) - int(freq_size/2) + patch_shift_freq)%patch_shift_freq)

nb_area_sky_ldev = int((map_pixel_size_ldev+2*orig_offset_sky_ldev)/patch_shift_sky)
nb_area_freq = int((map_pixel_freq_size+2*orig_offset_freq)/patch_shift_freq)



Writing /content/SDC2/config.py


### Training helper functions

In [6]:
%%writefile /content/SDC2/aux_fwd.py

from config import *

l_map_pixel_size = map_pixel_size_ldev + 2*orig_offset_sky_ldev
l_map_pixel_freq_size = map_pixel_freq_size + 2*orig_offset_freq

norm_data = np.memmap("DEV_norm_cube.bin", mode="r", dtype="uint16", shape=((l_map_pixel_freq_size, l_map_pixel_size, l_map_pixel_size)))

nb_inputs = nb_area_sky_ldev*nb_area_sky_ldev
inputs = np.zeros((nb_inputs,sky_size*sky_size*freq_size), dtype="float32")

def create_test_batch(patch_freq):


  for patch_dec in range(0,nb_area_sky_ldev):
    for patch_ra in range(0,nb_area_sky_ldev):

      i = patch_dec*nb_area_sky_ldev + patch_ra
      p_ra   = patch_ra*patch_shift_sky
      p_dec  = patch_dec*patch_shift_sky
      p_freq = patch_freq*patch_shift_freq

      patch = norm_data[p_freq:p_freq+freq_size, p_dec:p_dec+sky_size, p_ra:p_ra+sky_size]
      inputs[i,:] = (patch.flatten("C")/65535.0)*2.0 - 1.0

  return inputs


Writing /content/SDC2/aux_fwd.py


## 3 - Network prediction and post-processing


### Inference over the cube

In [ ]:
%%shell
cd /content/SDC2

python3 - <<EOF

from config import *
from aux_fwd import *

sys.path.insert(0,glob.glob('/content/CIANNA/src/build/lib.*/')[-1])
import CIANNA as cnn

cnn.init(in_dim=np.array([sky_size,sky_size,freq_size], dtype="int"), in_nb_ch=1, out_dim=0,
    bias=0.1, b_size=8, comp_meth='C_CUDA', dynamic_load=1,
    mixed_precision="FP16C_FP32A", inference_only=1, adv_size=30)

nb_yolo_filters = cnn.set_yolo_params(no_override=0, raw_output=0)

cnn.load("YOLO_CIANNA_net_model_SDC2_MC-BT2_MINERVA_Cornu2025.dat", 0, bin=1)

if(os.path.isfile("raw_pred.dat")):
  os.system("rm raw_pred.dat")

#Do the inference successively over independant frequency slices to avoid overloading Colab memory
for p_f in range(0, nb_area_freq):
  print ("Processing slice %d/%d"%(p_f, nb_area_freq))
  inputs_test = create_test_batch(p_f)
  nb_test = nb_area_sky_ldev*nb_area_sky_ldev
  targets_test = np.empty((0,0))

  cnn.create_dataset("TEST", nb_test, inputs_test[:,:], targets_test[:,:])
  cnn.forward(saving=2, no_error=1)
  cnn.delete_dataset("TEST")

  os.system("cat fwd_res/net0_0000.dat >> raw_pred.dat")

EOF

### Filtering, post-processing and conversion to catalog format


In [ ]:
%%writefile /content/SDC2/aux_post_proc.py

from config import *

#Distance IoU in 3D
@jit(nopython=True, cache=True, fastmath=False)
def fct_DIoU(box1, box2):
    inter_w = max(0.0, min(box1[3], box2[3]) - max(box1[0], box2[0]))
    inter_h = max(0.0, min(box1[4], box2[4]) - max(box1[1], box2[1]))
    inter_d = max(0.0, min(box1[5], box2[5]) - max(box1[2], box2[2]))

    inter_3d = inter_w * inter_h * inter_d
    uni_3d =  abs(box1[3]-box1[0])*abs(box1[4]-box1[1])*abs(box1[5]-box1[2]) \
            + abs(box2[3]-box2[0])*abs(box2[4]-box2[1])*abs(box2[5]-box2[2]) \
            - inter_3d
    enclose_w = (max(box1[3], box2[3]) - min(box1[0], box2[0]))
    enclose_h = (max(box1[4], box2[4]) - min(box1[1], box2[1]))
    enclose_d = (max(box1[5], box2[5]) - min(box1[2], box2[2]))

    cx_a = (box1[3] + box1[0])*0.5; cx_b = (box2[3] + box2[0])*0.5
    cy_a = (box1[4] + box1[1])*0.5; cy_b = (box2[4] + box2[1])*0.5
    cz_a = (box1[5] + box1[2])*0.5; cz_b = (box2[5] + box2[2])*0.5
    dist_cent = np.sqrt((cx_a - cx_b)*(cx_a - cx_b) + (cy_a - cy_b)*(cy_a - cy_b) + (cz_a - cz_b)*(cz_a - cz_b))
    diag_enclose = np.sqrt(enclose_w*enclose_w + enclose_h*enclose_h + enclose_d*enclose_d)

    return float(inter_3d)/float(uni_3d) - float(dist_cent)/float(diag_enclose)


@jit(nopython=True, cache=True, fastmath=False)
def tile_filter(c_pred, c_box, c_tile, nb_box, nb_param, yolo_nb_sky_reg, yolo_nb_freq_reg):
	c_nb_box = 0
	for p_f in range(0, yolo_nb_freq_reg):
		for p_dec in range(0,yolo_nb_sky_reg):
			for p_ra in range(0,yolo_nb_sky_reg):

				for k in range(0,nb_box):
					offset = int(k*(8+nb_param))
					c_box[6] = c_pred[offset+6,p_f,p_dec,p_ra] #probability
					c_box[7] = c_pred[offset+7,p_f,p_dec,p_ra] #objectness
					#Manual objectness penality on the edges of the images (help for both obj selection and NMS)
					if(p_ra == 0 or p_ra == yolo_nb_sky_reg-1 or \
					   p_dec == 0 or p_dec == yolo_nb_sky_reg-1 or\
					   p_f == 0 or p_f == yolo_nb_freq_reg-1):
						c_box[6] = max(0.03,c_box[6]-0.05)
						c_box[7] = max(0.03,c_box[7]-0.05)

					if(c_box[7] >= 0.1):
						bx = (c_pred[offset+0,p_f,p_dec,p_ra] + c_pred[offset+3,p_f,p_dec,p_ra])*0.5
						by = (c_pred[offset+1,p_f,p_dec,p_ra] + c_pred[offset+4,p_f,p_dec,p_ra])*0.5
						bz = (c_pred[offset+2,p_f,p_dec,p_ra] + c_pred[offset+5,p_f,p_dec,p_ra])*0.5
						bw = max(5.0, c_pred[offset+3,p_f,p_dec,p_ra] - c_pred[offset+0,p_f,p_dec,p_ra])
						bh = max(5.0, c_pred[offset+4,p_f,p_dec,p_ra] - c_pred[offset+1,p_f,p_dec,p_ra])
						bd = max(5.0, c_pred[offset+5,p_f,p_dec,p_ra] - c_pred[offset+2,p_f,p_dec,p_ra])

						c_box[0] = bx - bw*0.5; c_box[3] = bx + bw*0.5
						c_box[1] = by - bh*0.5; c_box[4] = by + bh*0.5
						c_box[2] = bz - bd*0.5; c_box[5] = bz + bd*0.5

						c_box[8] = k
						c_box[9:9+nb_param] = c_pred[offset+8:offset+8+nb_param,p_f,p_dec,p_ra]
						c_box[-1] = p_f*yolo_nb_freq_reg*yolo_nb_sky_reg + p_dec*yolo_nb_sky_reg + p_ra
						c_tile[c_nb_box,:] = c_box[:]
						c_nb_box += 1

	return c_nb_box


@jit(nopython=True, cache=True, fastmath=False)
def first_NMS(c_tile, c_tile_kept, c_box, c_nb_box, nms_threshold):
	c_nb_box_final = 0
	is_match = 1
	c_box_size_prev = c_nb_box

	while(c_nb_box > 0):
		max_objct = np.argmax(c_tile[:c_box_size_prev,7])
		c_box = np.copy(c_tile[max_objct])
		c_tile[max_objct,7] = 0.0
		c_tile_kept[c_nb_box_final] = c_box
		c_nb_box_final += 1; c_nb_box -= 1; i = 0

		for i in range(0,c_box_size_prev):
			if(c_tile[i,7] < 0.0000000001):
				continue
			IoU = fct_DIoU(c_box[:6], c_tile[i,:6])
			if(IoU >  nms_threshold):
				c_tile[i,7] = 0.0
				c_nb_box -= 1

	return c_nb_box_final


@jit(nopython=True, cache=True, fastmath=False)
def inter_patch_NMS(boxes, comp_boxes, c_tile, direction, overlap, patch_shift, patch_size, nms_threshold):
	c_tile[:,:] = 0.0
	nb_box_kept = 0

	mask_keep = np.where((boxes[:,0] > overlap[0]) & (boxes[:,3] < patch_shift[0]) &\
						 (boxes[:,1] > overlap[1]) & (boxes[:,4] < patch_shift[1]) &\
						 (boxes[:,2] > overlap[2]) & (boxes[:,5] < patch_shift[2]))[0]
	mask_remain = np.where((boxes[:,0] <= overlap[0]) | (boxes[:,3] >= patch_shift[0]) |\
						   (boxes[:,1] <= overlap[1]) | (boxes[:,4] >= patch_shift[1]) |\
						   (boxes[:,2] <= overlap[2]) | (boxes[:,5] >= patch_shift[2]))[0]

	nb_box_kept = np.shape(mask_keep)[0]
	c_tile[0:nb_box_kept,:] = boxes[mask_keep,:]
	comp_boxes[:,0:3] += direction[:]*patch_shift[:]
	comp_boxes[:,3:6] += direction[:]*patch_shift[:]

	comp_mask_keep = np.where((comp_boxes[:,0] < patch_size[0]) & (comp_boxes[:,3] > 0) &\
							  (comp_boxes[:,1] < patch_size[1]) & (comp_boxes[:,4] > 0) &\
							  (comp_boxes[:,2] < patch_size[2]) & (comp_boxes[:,5] > 0))[0]

	for b_ref in mask_remain:
		found = 0
		for b_comp in comp_mask_keep:
			IoU = fct_DIoU(boxes[b_ref,:6], comp_boxes[b_comp,:6])
			#print(IoU, boxes[b_ref,7], comp_boxes[b_comp,7])
			if(IoU > nms_threshold and boxes[b_ref,7] < comp_boxes[b_comp,7]):
				found = 1
				break
		if(found == 0):
			c_tile[nb_box_kept,:] = boxes[b_ref,:]
			nb_box_kept += 1

	return nb_box_kept


In [ ]:
%cd /content/SDC2

from config import *
from aux_fwd import *
from aux_post_proc import *

pred_data = np.fromfile("raw_pred.dat", dtype="float32")
pred_data = np.reshape(pred_data, (nb_area_freq, nb_area_sky_ldev, nb_area_sky_ldev, nb_box*(8+nb_param), yolo_nb_freq_reg, yolo_nb_sky_reg, yolo_nb_sky_reg))

pred_boxes = []

c_tile = np.zeros((yolo_nb_sky_reg*yolo_nb_sky_reg*yolo_nb_freq_reg*nb_box,(8+1+nb_param+1)),dtype="float32")
c_tile_kept = np.zeros((yolo_nb_sky_reg*yolo_nb_sky_reg*yolo_nb_freq_reg*nb_box,(8+1+nb_param+1)),dtype="float32")
c_box = np.zeros((8+1+nb_param+1),dtype="float32")

total_n_box = 0
for p_freq in range(0,nb_area_freq):
	for p_dec in range(0,nb_area_sky_ldev):
		for p_ra in range(0,nb_area_sky_ldev):

			c_tile[:,:] = 0.0
			c_tile_kept[:,:] = 0.0
			c_pred = pred_data[p_freq,p_dec,p_ra,:,:,:]

			c_nb_box_final = tile_filter(c_pred, c_box, c_tile, nb_box, nb_param, yolo_nb_sky_reg, yolo_nb_freq_reg)
			c_nb_box_final = first_NMS(c_tile, c_tile_kept, c_box, c_nb_box_final, -0.1)

			total_n_box += c_nb_box_final
			pred_boxes.append(np.copy(c_tile_kept[0:c_nb_box_final]))

if(total_n_box <= 1):
	print ("No prediction found, closing fwd. Final score: 0")
	exit()
else:
	pred_boxes = np.reshape(np.array(pred_boxes, dtype="object"), (nb_area_freq, nb_area_sky_ldev, nb_area_sky_ldev))

c_tile = np.zeros((yolo_nb_sky_reg*yolo_nb_sky_reg*yolo_nb_freq_reg*nb_box,(8+1+nb_param+1)),dtype="float32")

A = np.array([-1, 0, 1]); B = np.array([-1, 0, 1]); C = np.array([-1, 0, 1])
x, y, z = np.meshgrid(A, B, C)
dir_array = np.vstack([x.ravel(), y.ravel(), z.ravel()]).T

l_overlap = np.array((overlap_sky, overlap_sky, overlap_freq))
l_patch_shift = np.array((patch_shift_sky, patch_shift_sky, patch_shift_freq))
l_patch_size = np.array((sky_size, sky_size, freq_size))

#Second NMS over all the overlapping patches
for p_freq in range(0,nb_area_freq):
	for p_dec in range(0,nb_area_sky_ldev):
		for p_ra in range(0,nb_area_sky_ldev):
			boxes = np.copy(pred_boxes[p_freq,p_dec,p_ra])
			for l in range(0,np.shape(dir_array)[0]):
				if(p_freq+dir_array[l,2] >= 0 and p_freq+dir_array[l,2] <= nb_area_freq-1 and\
				   p_dec +dir_array[l,1] >= 0 and p_dec +dir_array[l,1] <= nb_area_sky_ldev-1  and\
				   p_ra  +dir_array[l,0] >= 0 and p_ra  +dir_array[l,0] <= nb_area_sky_ldev-1 ):
					comp_boxes = np.copy(pred_boxes[p_freq+dir_array[l,2],p_dec+dir_array[l,1],p_ra+dir_array[l,0]])
					c_nb_box = inter_patch_NMS(boxes, comp_boxes, c_tile, dir_array[l], l_overlap, l_patch_shift, l_patch_size, -0.3)
					boxes = np.copy(c_tile[0:c_nb_box,:])

			pred_boxes[p_freq,p_dec,p_ra] = np.copy(boxes)

#Convert boxes from per-input coordinates to original cube pixel coordinates
for p_freq in range(0,nb_area_freq):
	box_freq_offset = p_freq*patch_shift_freq
	for p_dec in range(0,nb_area_sky_ldev):
		box_dec_offset = p_dec*patch_shift_sky
		for p_ra in range(0,nb_area_sky_ldev):
			box_ra_offset = p_ra*patch_shift_sky

			pred_boxes[p_freq,p_dec,p_ra][:,0] = box_ra_offset   + pred_boxes[p_freq,p_dec,p_ra][:,0] - orig_offset_sky_ldev - 0.5
			pred_boxes[p_freq,p_dec,p_ra][:,1] = box_dec_offset  + pred_boxes[p_freq,p_dec,p_ra][:,1] - orig_offset_sky_ldev - 0.5
			pred_boxes[p_freq,p_dec,p_ra][:,2] = box_freq_offset + pred_boxes[p_freq,p_dec,p_ra][:,2] - orig_offset_freq     - 0.5
			pred_boxes[p_freq,p_dec,p_ra][:,3] = box_ra_offset   + pred_boxes[p_freq,p_dec,p_ra][:,3] - orig_offset_sky_ldev - 0.5
			pred_boxes[p_freq,p_dec,p_ra][:,4] = box_dec_offset  + pred_boxes[p_freq,p_dec,p_ra][:,4] - orig_offset_sky_ldev - 0.5
			pred_boxes[p_freq,p_dec,p_ra][:,5] = box_freq_offset + pred_boxes[p_freq,p_dec,p_ra][:,5] - orig_offset_freq     - 0.5

#Merge predicted box list from all input regions
box_cat = np.vstack(pred_boxes.flatten())
box_cat = box_cat[box_cat[:,7].argsort(),:][::-1]

#Save filtered box catalog in detector format ordered by objectness
np.savetxt("pred_ldev_cat_filtered_repos_ordered.dat", box_cat)

#Load cube WCS and convert box pixel (x,y) coordinates to (RA,DEC)
hdul_ldev = fits.open("cont_dev.fits", memmap=True)
wcs_ldev = WCS(hdul_ldev[0].header)

cls = utils.pixel_to_skycoord((box_cat[:,3]+box_cat[:,0])*0.5, (box_cat[:,4]+box_cat[:,1])*0.5, wcs_ldev)
ra_dec_coords = np.array([cls.ra.deg, cls.dec.deg])

#Convert all predicted quantities to the SDC2 source catalog format
cat_header = "id ra dec hi_size line_flux_integral central_freq pa i w20"
cat_size = int(np.shape(box_cat)[0])
final_box_cat = np.zeros((cat_size,9), dtype="float32")

lims = np.loadtxt("MC-BT2_train_cat_lims.txt")

final_box_cat[:,0] = np.arange(0,cat_size)
final_box_cat[:,[1,2]] = ra_dec_coords.T
final_box_cat[:,3] = box_cat[:,10]*lims[1,0] + lims[1,1]
final_box_cat[:,4] = np.exp(box_cat[:,9]*lims[0,0] + lims[0,1])
final_box_cat[:,5] = (box_cat[:,5]+box_cat[:,2])*0.5*pixel_size_freq + 9.5e8
final_box_cat[:,6] = np.mod(np.arctan2(np.clip(box_cat[:,12],0.0,1.0)*2.0-1.0, np.clip(box_cat[:,13],0.0,1.0)*2.0-1.0)*180.0/np.pi,360.0)
final_box_cat[:,7] = np.arccos(np.clip(box_cat[:,14],0.0,1.0))*180.0/np.pi
final_box_cat[:,8] = (np.exp(box_cat[:,11]*lims[2,0] + lims[2,1])*pixel_size_freq)/(final_box_cat[:,5]**2/1.4204e9)*299792.458

np.savetxt("pred_ldev_final_catalog.txt", final_box_cat, header=cat_header, comments="", fmt="%d %3.13f %2.13f %1.13f %1.13f %10.1f %3.13f %2.13f %3.13f")


## 4- Scoring the predicted catalog

#### Installing SDC2 scorer pipeline

In [ ]:
%%shell
pip install ska_sdc

#DEV truth cat
wget --content-disposition https://www.dropbox.com/scl/fo/20mzfxit0yc9djr7n56v0/AHt8EbMSCx-b7jmbAxNrdqM/sky_dev_truthcat_v2.txt?rlkey=mstsccclw3euwq9g71ofywjo9

#### Optimizing selection threshold for the DEV cube based on the scorer

In [ ]:

from ska_sdc import Sdc2Scorer

min_obj = 0.65 #Apply a permissive objectness filtering by default to reduce the optimization range
max_size = np.shape(final_box_cat)[0] - np.searchsorted(box_cat[:,7][::-1], min_obj)
pred_cat = final_box_cat[:max_size]
raw_cat = box_cat[:max_size]

#First scoring to extract per source score for threshold optimization
np.savetxt("pre_opt_cat.txt", pred_cat, header=cat_header, comments="", fmt="%d %3.13f %2.13f %1.13f %1.13f %10.1f %3.13f %2.13f %3.13f")

sub_cat_path = "pre_opt_cat.txt"
truth_cat_path = "sky_dev_truthcat_v2.txt"

scorer = Sdc2Scorer.from_txt(sub_cat_path, truth_cat_path, sub_skiprows=0, truth_skiprows=0)
scorer.run(detail=True)
score_details = scorer.score.scores_df

per_source_score = np.zeros((max_size)) - 1.0
per_source_score[score_details["id"]] = score_details.to_numpy()[:,1:].sum(axis=1)/7.0

nb_freq_bins = 20
nb_obj_bins = 35
optimized_catalog = []

for k in range(0, nb_freq_bins):
	index_c_freq = np.where((pred_cat[:,5] > 9.5e8 + k*2e8/nb_freq_bins) & (pred_cat[:,5] < 9.5e8 + (k+1)*2e8/nb_freq_bins))[0]
	l_pred = pred_cat[index_c_freq]
	l_obj = raw_cat[index_c_freq,7]
	l_score = per_source_score[index_c_freq]

	dig_bins = np.logspace(np.log10(min_obj),0,num=nb_obj_bins+1)
	dig_index = np.digitize(l_obj, bins=dig_bins, right=True)

	opt_array = np.zeros((nb_obj_bins,4))
	for l in range(0,nb_obj_bins):
		bin_object_id = np.where((dig_index[:] == l))[0]
		nb_tot_bin = int(np.shape(bin_object_id)[0])
		match_id = np.where(l_score[bin_object_id] > 0.0)[0]
		nb_match = np.shape(match_id)[0]

		avg_score = 0; l_purity = 0
		if(nb_match > 0):
			avg_score = np.mean(l_score[bin_object_id])
		if(nb_tot_bin > 0):
			l_purity = nb_match/nb_tot_bin
		add_score = np.sum(l_score[bin_object_id[match_id]]) - (nb_tot_bin-nb_match)

		opt_array[l,:] = [nb_tot_bin, l_purity, avg_score, add_score]

	for l in range(0,nb_obj_bins-1):
		if(np.all(np.cumsum(opt_array[l:,3]) > 0)):
			id_opt = l
			break
	opt_bin_obj_select = dig_bins[l-1]
	max_opt_size = np.shape(l_obj)[0] - np.searchsorted(l_obj[::-1], opt_bin_obj_select)
	optimized_catalog.append(l_pred[:max_opt_size])

optimized_catalog = np.vstack(optimized_catalog)
np.savetxt("pred_dev_final_catalog_optimized.txt", optimized_catalog, header=cat_header, comments="", fmt="%d %3.13f %2.13f %1.13f %1.13f %10.1f %3.13f %2.13f %3.13f")



#### Final scoring over the DEV truthcat

In [ ]:

from ska_sdc import Sdc2Scorer

#Final scoring from the optimized predicted source catalog
sub_cat_path = "pred_dev_final_catalog_optimized.txt"
truth_cat_path = "sky_dev_truthcat_v2.txt"

scorer = Sdc2Scorer.from_txt(sub_cat_path, truth_cat_path, sub_skiprows=0, truth_skiprows=0)
scorer.run(detail=True)

print("Final score: {}".format(scorer.score.value))
print ("Ndet:", scorer.score.n_det, "\nNmatch:", scorer.score.n_match, "\nNfalse:", scorer.score.n_bad + scorer.score.n_false)
print ("Avg characterization score:",scorer.score.acc_pc, "\nPurity:", (scorer.score.n_match/(scorer.score.n_match+scorer.score.n_false)))

score_details = scorer.score.scores_df

print ("Avg subscores:")
print ("Position:",np.mean(score_details["position"]))
print ("Central frequency:",np.mean(score_details["central_freq"]))
print ("Line flux", np.mean(score_details["flux"]))
print ("HI size",np.mean(score_details["hi_size"]))
print ("Line width",np.mean(score_details["w20"]))
print ("PA",np.mean(score_details["pa"]))
print ("Inclination",np.mean(score_details["i"]))

##  5 - Scoring of of other catalogs

This following cells download all truth catalog and all catalog from Cornu et al. 2025 for individual scoring. This is intended to verify the resuts and help with reproductibility.


### Downloading models and truth catalogs

In [ ]:
%%shell

#LDEV truth cat
if [ ! -f "sky_ldev_truthcat_v2.txt" ]; then
wget --content-disposition https://www.dropbox.com/scl/fo/e847a6pnjtqk7xmxz6flt/AIo4631BbKfwhW2NGhvtbkw/sky_ldev_truthcat_v2.txt?rlkey=84kkeaw021ajh7t9lqur2n7p8
fi
#MAIN truth cat
if [ ! -f "sky_full_truthcat_v2.txt" ]; then
wget --content-disposition https://www.dropbox.com/scl/fo/ce4o2tqhy2ddkowwecs5z/ADaaQzoKGwkHRkdPLZURw1M/sky_full_truthcat_v2.txt?rlkey=ywcljxh6eq2q18r629prn5w0j
fi

#Get the complete models archivey
if [ ! -f "Cornu_et_al_2025_SDC2_models_catalogs_and_codes_archive.zip" ]; then
wget --content-disposition https://share.obspm.fr/s/swyCT7BgEGjtZK3/download/Cornu_et_al_2025_SDC2_models_catalogs_and_codes_archive
fi
unzip -o Cornu_et_al_2025_SDC2_models_catalogs_and_codes_archive.zip

#### Computing scores

In [ ]:
from ska_sdc import Sdc2Scorer

#Final scoring from the optimized predicted source catalog
sub_cat_path = "Cornu_et_al_2025_SDC2_models_catalogs_and_codes_archive/catalogs/YOLO_CIANNA_catalog_SDC2_BT1_sc25453_MINERVA_Cornu2025.txt"
truth_cat_path = "sky_full_truthcat_v2.txt"


scorer = Sdc2Scorer.from_txt(sub_cat_path, truth_cat_path, sub_skiprows=0, truth_skiprows=0)
scorer.run(detail=True)

print("Final score: {}".format(scorer.score.value))
print ("Ndet:", scorer.score.n_det, "\nNmatch:", scorer.score.n_match, "\nNfalse:", scorer.score.n_bad + scorer.score.n_false)
print ("Avg characterization score:",scorer.score.acc_pc, "\nPurity:", (scorer.score.n_match/(scorer.score.n_match+scorer.score.n_false)))

score_details = scorer.score.scores_df

print ("Avg subscores:")
print ("Position:",np.mean(score_details["position"]))
print ("Central frequency:",np.mean(score_details["central_freq"]))
print ("Line flux", np.mean(score_details["flux"]))
print ("HI size",np.mean(score_details["hi_size"]))
print ("Line width",np.mean(score_details["w20"]))
print ("PA",np.mean(score_details["pa"]))
print ("Inclination",np.mean(score_details["i"]))

#### Target and predicted sources flux distribution


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as colors

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    if isinstance(cmap, str):
        cmap = plt.get_cmap(cmap)
    new_cmap = colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap

cmap = truncate_colormap("terrain_r",0.08,1.0)

pred_cat = np.loadtxt(sub_cat_path, skiprows=1)
#id ra dec hi_size line_flux_integral central_freq pa i w20 snr_unsmoothed snr_smoothed
truth_cat = np.loadtxt(truth_cat_path, skiprows=1)
match_df = scorer.score.match_df
false_det = np.delete(pred_cat, match_df["id"], axis=0)

completeness_min_flux = 18

bins_x = 10**np.linspace(0, 3.0,56)
bins_y = 10**np.linspace(0.4, 5,50)
rwidth = 0.88

fig, axs = plt.subplots(2,1, figsize=(7,5), dpi=150, constrained_layout=True, gridspec_kw={"height_ratios":[1.0,1.0]})

n, bins1, patches1 = axs[0].hist(truth_cat[:,4], bins=bins_x, histtype="bar", color = "grey", rwidth = rwidth,
        log=True, label="True cat.", edgecolor='white', linewidth=0.0, rasterized=True)

n2, bins2, patches2 = axs[0].hist(np.append(match_df["line_flux_integral_t"],false_det[:,4]), bins=bins_x, histtype="bar",
        color = "gold", alpha=0.85, rwidth = rwidth, log=True, label="True+False det.", edgecolor='white', linewidth=0.0, rasterized=True)

n3, bins3, patches3 = axs[0].hist(match_df["line_flux_integral_t"], bins=bins_x, histtype="bar", color = "lightgrey", rwidth = rwidth,
        log=True, label="True det.", edgecolor='white', linewidth=0.0, rasterized=True)

axs[0].plot([completeness_min_flux,completeness_min_flux],[0,max(n)*3], ls="--", color="red")
axs[0].set_xscale('log')
axs[0].xaxis.set_minor_locator(ticker.LogLocator(numticks=999, subs="auto"))
axs[0].set_xlim(1e0,1e3)
axs[0].set_ylim(0,max(n)*3.0)
axs[0].tick_params(axis='both', which='major', labelsize=14)
axs[0].set_ylabel("N. of objects", fontsize=16)
axs[0].tick_params(labelbottom=False)
axs[0].text(0.98,0.95,"Matches target flux", color="k", fontsize=12,
        transform=axs[0].transAxes, va="top", ha="right", style="italic")
ax2 = axs[0].twinx()
completeness_min_bin = np.searchsorted(bins1, completeness_min_flux) - 1

ax2.step((bins1[:-1]+bins1[1:])*0.5, np.clip(n3/n,0,1), where="mid", c="C1", label="Completeness")
ax2.set_ylim(-0.05,1.5)
ax2.set_yticks([0.0,0.2,0.4,0.6,0.8,1.0])
ax2.tick_params(axis='y', labelcolor="C1", labelsize=12)
ax2.set_ylabel("Completeness", c="C1", loc="bottom", fontsize=12)


n, bins1, patches1 = axs[1].hist(truth_cat[:,4], bins=bins_x, histtype="bar", color = "grey", rwidth = rwidth,
        log=True, label="True cat.", edgecolor='white', linewidth=0.0, rasterized=True)

n2, bins2, patches2 = axs[1].hist(pred_cat[:,4], bins=bins_x, histtype="bar",
        color = "gold", alpha=0.85, rwidth = rwidth, log=True, label="True+False det.", edgecolor='white', linewidth=0.0, rasterized=True)

n3, bins3, patches3 = axs[1].hist(match_df["line_flux_integral"], bins=bins_x, histtype="bar", color = "lightgrey", rwidth = rwidth,
        log=True, label="True det.", edgecolor='white', linewidth=0.0, rasterized=True)

axs[1].plot([completeness_min_flux,completeness_min_flux],[0,max(n)*3], ls="--", color="red")
axs[1].set_xscale('log')
axs[1].xaxis.set_minor_locator(ticker.LogLocator(numticks=999, subs="auto"))
axs[1].set_xlim(1e0,1e3)
axs[1].set_ylim(0,max(n)*3.0)
axs[1].tick_params(axis='both', which='major', labelsize=14)
axs[1].set_ylabel("N. of objects", fontsize=16)
axs[1].set_xlabel("Integrated line flux [Jy Hz]", fontsize=16)
axs[1].text(0.98,0.95,"Matches predicted flux", color="k", fontsize=12,
        transform=axs[1].transAxes, va="top", ha="right", style="italic")
ax2 = axs[1].twinx()
completeness_min_bin = np.searchsorted(bins1, completeness_min_flux) - 2

ax2.step((bins1[completeness_min_bin:-1]+bins1[completeness_min_bin+1:])*0.5,
    np.clip(n3[completeness_min_bin:]/n2[completeness_min_bin:], 0, 1),
    where="mid", c="C0", label="Purity")
ax2.set_ylim(-0.05,1.5)
ax2.set_yticks([0.0,0.2,0.4,0.6,0.8,1.0])
ax2.tick_params(axis='y', labelcolor="C0", labelsize=12)
ax2.set_ylabel("Purity", c="C0", loc="bottom", fontsize=12)

plt.show()